In [ ]:
from functools import reduce
from pyspark.sql import functions as F


def read_bronze(table: str):
    for name in (f"lh_bronze.dbo.{table}", f"lh_bronze.{table}"):
        try:
            return spark.table(name)
        except Exception:
            continue
    raise ValueError(f"Could not read {table} from lh_bronze")


def num(c):
    return F.col(c).cast("double")


def blank(c):
    return F.col(c).isNull() | (F.trim(F.col(c)) == "")


def cnt(condition):
    return F.sum(F.when(condition, 1).otherwise(0))


def bad_hhmm(c):
    v = num(c)
    return (~blank(c)) & (v.isNull() | (v < 0) | (v > 2400) | ((v % 100) >= 60))


bronze = read_bronze("bronze_ontime")

required = [
    "flight_date", "reporting_airline", "flight_number_reporting_airline", "tail_number",
    "origin", "dest", "crs_dep_time", "dep_time", "dep_delay", "crs_arr_time", "arr_time",
    "arr_delay", "arr_delay_minutes", "arr_del15", "cancelled", "cancellation_code",
    "diverted", "air_time", "actual_elapsed_time", "distance", "flights",
    "carrier_delay", "weather_delay", "nas_delay", "security_delay", "late_aircraft_delay",
]

print("=== A. Required columns ===")
missing = [c for c in required if c not in bronze.columns]
print(f"Missing: {missing or 'none'}")
if missing:
    print(f"Available columns: {bronze.columns}")
    raise ValueError("Column names differ from expected. Stopping.")

p = (
    bronze.select("_period_key", *required)
    .withColumn("is_cancelled", num("cancelled") == 1)
    .withColumn("is_diverted", num("diverted") == 1)
    .withColumn("is_operated", (num("cancelled") == 0) & (num("diverted") == 0))
)

print("\n=== B. Flight status for key months ===")
sample_months = ["2019-07", "2020-03", "2020-04", "2020-05", "2024-07"]
for r in (
    p.filter(F.col("_period_key").isin(sample_months))
    .groupBy("_period_key")
    .agg(
        F.count("*").alias("rows"),
        cnt(F.col("is_cancelled")).alias("cancelled"),
        cnt(F.col("is_diverted")).alias("diverted"),
        cnt(F.col("is_operated")).alias("operated"),
    )
    .orderBy("_period_key")
    .collect()
):
    print(
        f"{r['_period_key']}: rows {r['rows']:,} | cancelled {r['cancelled']:,} "
        f"({r['cancelled'] / r['rows']:.1%}) | diverted {r['diverted']:,} | operated {r['operated']:,}"
    )

op = F.col("is_operated")
not_cancelled = ~F.col("is_cancelled")
delayed_15 = op & (num("arr_del15") == 1)
cause_cols = ["carrier_delay", "weather_delay", "nas_delay", "security_delay", "late_aircraft_delay"]
cause_sum = reduce(lambda a, b: a + b, [F.coalesce(num(c), F.lit(0.0)) for c in cause_cols])
all_causes_blank = reduce(lambda a, b: a & b, [blank(c) for c in cause_cols])
key_fields = [
    "flight_date", "reporting_airline", "flight_number_reporting_airline", "tail_number",
    "origin", "dest", "crs_dep_time", "crs_arr_time",
]
time_fields = ["crs_dep_time", "dep_time", "crs_arr_time", "arr_time"]

metrics = [
    F.count("*").alias("rows"),
    cnt(F.col("is_cancelled")).alias("cancelled"),
    cnt(F.col("is_diverted")).alias("diverted"),
    cnt(op).alias("operated"),
    cnt(F.col("is_cancelled") & F.col("is_diverted")).alias("cancelled_and_diverted"),
    cnt(num("cancelled").isNull() | ~num("cancelled").isin(0, 1)).alias("invalid_cancelled_flag"),
    cnt(num("diverted").isNull() | ~num("diverted").isin(0, 1)).alias("invalid_diverted_flag"),
    cnt(not_cancelled & blank("dep_time")).alias("not_cancelled_missing_dep_time"),
    cnt(op & blank("arr_delay")).alias("operated_missing_arr_delay"),
    cnt(op & blank("air_time")).alias("operated_missing_air_time"),
    F.min(F.when(op, num("dep_delay"))).alias("min_dep_delay"),
    F.max(F.when(op, num("dep_delay"))).alias("max_dep_delay"),
    F.min(F.when(op, num("arr_delay"))).alias("min_arr_delay"),
    F.max(F.when(op, num("arr_delay"))).alias("max_arr_delay"),
    cnt(op & (num("arr_delay") > 1440)).alias("arr_delay_over_24h"),
    F.min(F.when(op, num("distance"))).alias("min_distance"),
    F.max(F.when(op, num("distance"))).alias("max_distance"),
    cnt(op & (num("air_time") <= 0)).alias("air_time_zero_or_negative"),
    cnt(op & (num("actual_elapsed_time") <= 0)).alias("elapsed_time_zero_or_negative"),
    cnt(op & (num("air_time") > num("actual_elapsed_time"))).alias("air_time_exceeds_elapsed"),
    cnt(delayed_15).alias("delayed_15_flights"),
    cnt(delayed_15 & all_causes_blank).alias("delayed_15_no_causes"),
    cnt(delayed_15 & (F.abs(cause_sum - num("arr_delay_minutes")) <= 1)).alias("delayed_15_causes_match"),
    cnt(~op & ~all_causes_blank).alias("not_operated_with_causes"),
]
metrics += [cnt(blank(c)).alias(f"blank_{c}") for c in key_fields]
for c in time_fields:
    metrics += [
        cnt(bad_hhmm(c)).alias(f"invalid_{c}"),
        cnt(num(c) == 2400).alias(f"value_2400_{c}"),
        cnt((~blank(c)) & (F.length(F.trim(F.col(c))) < 4)).alias(f"short_{c}"),
    ]

m = p.agg(*metrics).first().asDict()

print("\n=== C. Overall status and flag quality ===")
print(f"Rows: {m['rows']:,}")
print(f"Cancelled: {m['cancelled']:,} ({m['cancelled'] / m['rows']:.2%})")
print(f"Diverted: {m['diverted']:,} ({m['diverted'] / m['rows']:.2%})")
print(f"Operated: {m['operated']:,}")
print(f"Cancelled AND diverted: {m['cancelled_and_diverted']:,}")
print(f"Invalid cancelled flag: {m['invalid_cancelled_flag']:,}")
print(f"Invalid diverted flag: {m['invalid_diverted_flag']:,}")
print("Distinct values in 'flights' column: " + ", ".join(
    f"{r['flights']} ({r['count']:,})" for r in p.groupBy("flights").count().collect()
))

print("\n=== D. Missing key fields ===")
for c in key_fields:
    print(f"Blank {c}: {m[f'blank_{c}']:,}")
print(f"Not cancelled but missing dep_time: {m['not_cancelled_missing_dep_time']:,}")
print(f"Operated but missing arr_delay: {m['operated_missing_arr_delay']:,}")
print(f"Operated but missing air_time: {m['operated_missing_air_time']:,}")

print("\n=== E. Time formats ===")
for c in time_fields:
    print(
        f"{c}: invalid {m[f'invalid_{c}']:,} | equals 2400 {m[f'value_2400_{c}']:,} "
        f"| fewer than 4 digits {m[f'short_{c}']:,}"
    )

print("\n=== F. Value ranges (operated flights) ===")
print(f"Departure delay: {m['min_dep_delay']} to {m['max_dep_delay']} minutes")
print(f"Arrival delay: {m['min_arr_delay']} to {m['max_arr_delay']} minutes")
print(f"Arrival delay over 24 hours: {m['arr_delay_over_24h']:,}")
print(f"Distance: {m['min_distance']} to {m['max_distance']} miles")
print(f"Air time zero or negative: {m['air_time_zero_or_negative']:,}")
print(f"Elapsed time zero or negative: {m['elapsed_time_zero_or_negative']:,}")
print(f"Air time greater than elapsed time: {m['air_time_exceeds_elapsed']:,}")

print("\n=== G. Delay causes ===")
d15 = m["delayed_15_flights"]
print(f"Operated flights arriving 15+ min late: {d15:,}")
print(f"  with no cause values: {m['delayed_15_no_causes']:,}")
print(f"  where causes sum to total delay (+/- 1 min): {m['delayed_15_causes_match']:,} ({m['delayed_15_causes_match'] / d15:.2%})")
print(f"Cancelled or diverted flights with cause values: {m['not_operated_with_causes']:,}")

print("\n=== H. Cancellation reasons and duplicates ===")
print("Cancellation codes: " + ", ".join(
    f"{r['cancellation_code']} ({r['count']:,})"
    for r in p.filter(F.col("is_cancelled")).groupBy("cancellation_code").count().orderBy("cancellation_code").collect()
))

dup_key = ["flight_date", "reporting_airline", "flight_number_reporting_airline", "origin", "dest", "crs_dep_time"]
dups = p.groupBy(*dup_key).count().filter(F.col("count") > 1).cache()
dup_summary = dups.agg(F.count("*").alias("groups"), F.sum("count").alias("rows")).first()
dup_groups = dup_summary["groups"] or 0
dup_rows = dup_summary["rows"] or 0
print(f"Duplicate flight keys: {dup_groups:,} keys covering {dup_rows:,} rows ({dup_rows - dup_groups:,} extra rows)")
if dup_groups:
    for r in dups.orderBy(F.desc("count")).limit(3).collect():
        print(f"  Example: {[r[c] for c in dup_key]} appears {r['count']} times")
dups.unpersist()

=== A. Required columns ===
Missing: none

=== B. Flight status for key months ===
2019-07: rows 659,029 | cancelled 12,928 (2.0%) | diverted 2,320 | operated 643,781
2020-03: rows 648,229 | cancelled 110,187 (17.0%) | diverted 797 | operated 537,245
2020-04: rows 313,382 | cancelled 130,078 (41.5%) | diverted 298 | operated 183,006
2020-05: rows 180,617 | cancelled 11,613 (6.4%) | diverted 357 | operated 168,647
2024-07: rows 634,613 | cancelled 18,372 (2.9%) | diverted 2,303 | operated 613,938

=== C. Overall status and flag quality ===
Rows: 49,251,865
Cancelled: 1,063,521 (2.16%)
Diverted: 119,546 (0.24%)
Operated: 48,068,798
Cancelled AND diverted: 0
Invalid cancelled flag: 0
Invalid diverted flag: 0
Distinct values in 'flights' column: 1.00 (49,251,865)

=== D. Missing key fields ===
Blank flight_date: 0
Blank reporting_airline: 0
Blank flight_number_reporting_airline: 1
Blank tail_number: 282,043
Blank origin: 0
Blank dest: 0
Blank crs_dep_time: 0
Blank crs_arr_time: 0
Not cance

In [ ]:
from pyspark.sql import functions as F

codes_by_year = (
    bronze
    .filter(F.col("cancelled").cast("double") == 1)
    .withColumn("yr", F.substring("_period_key", 1, 4))
    .groupBy("yr")
    .pivot("cancellation_code", ["A", "B", "C", "D"])
    .count()
    .fillna(0)
    .orderBy("yr")
    .collect()
)

print("Cancellations by year and reason (A=Airline, B=Weather, C=Air traffic system, D=Security)")
for r in codes_by_year:
    total = r["A"] + r["B"] + r["C"] + r["D"]
    print(
        f"{r['yr']}: total {total:,} | A {r['A']:,} | B {r['B']:,} | "
        f"C {r['C']:,} | D {r['D']:,} ({r['D'] / total:.1%} security)"
    )

Cancellations by year and reason (A=Airline, B=Weather, C=Air traffic system, D=Security)
2019: total 134,925 | A 37,854 | B 72,277 | C 24,775 | D 19 (0.0% security)
2020: total 281,034 | A 19,008 | B 20,424 | C 4,413 | D 237,189 (84.4% security)
2021: total 103,060 | A 40,768 | B 53,820 | C 7,622 | D 850 (0.8% security)
2022: total 181,286 | A 71,780 | B 93,013 | C 15,469 | D 1,024 (0.6% security)
2023: total 87,943 | A 25,336 | B 47,923 | C 14,472 | D 212 (0.2% security)
2024: total 96,315 | A 30,926 | B 53,605 | C 11,780 | D 4 (0.0% security)
2025: total 102,876 | A 21,073 | B 63,404 | C 18,349 | D 50 (0.0% security)
2026: total 76,082 | A 14,262 | B 55,355 | C 6,391 | D 74 (0.1% security)
